# 02 - Behavioral data  (PLAN.md 5.1, gate V2)

**Hours 4-8. R1 lives here.**

~5000 MMLU items (primary) and TriviaQA (secondary, for the 5.4 transfer check),
across the **five fixed prompt conditions**. Record per item: answer, correctness,
verbalized confidence, and residual activations at the answer-adjacent position
across a layer sweep.

| # | Condition | Purpose |
|---|---|---|
| P1 | Separate - answer, then confidence in a fresh turn | the 2603.25052 baseline |
| P2 | Joint - answer and confidence in one pass | reasoning-contamination arm |
| P3 | Introspective - "how confident, and how do you know?" | invites self-report |
| P4 | Third-person - "how likely is *a model* to get this right?" | decouples report from self-model |
| P5 | Deferred - confidence after a filler turn | tests the cached answer-adjacent signal |

> **R9.** These five are fixed. Adding a sixth after seeing results invalidates
> the pre-registration in PLAN.md section 7. A post-hoc condition is a *follow-up on
> fresh data*, reported as exploratory.

**V2 is a hard gate:** verbalized confidence must have non-degenerate spread. If
the model says "80%" to everything, the whole project has no dependent variable
and you escalate to `NANDA_PRESET=escalate` (12b-it) per R1.


In [ ]:
# --- standard header ------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys, pathlib
for p in ("/workspace/NandaProj", ".."):
    if p not in sys.path:
        sys.path.insert(0, p)

from nandaproj import config

cfg = config.get_model_config()      # NANDA_PRESET env var, defaults to debug
config.ensure_dirs()
print("preset:", cfg.name, "|", cfg.n_params, "|", cfg.dtype)
print("device:", config.get_device())
print("results ->", config.RESULTS)


preset: google/gemma-3-270m-it | 270M | bfloat16
device: cuda
results -> /workspace/results


In [ ]:
def gate(name, ok, detail=""):
    """PLAN.md section 6 verification gate. Fails loudly and stops the notebook."""
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}  {detail}")
    if not ok:
        raise AssertionError(f"gate {name} failed: {detail}")


In [ ]:
# The five conditions, defined once, in one place, and not edited after results
# are seen. Keeping them as data (not scattered f-strings) is what makes the
# "fixed in advance" claim checkable later.
CONDITIONS = ["P1_separate", "P2_joint", "P3_introspective",
              "P4_third_person", "P5_deferred"]

# TODO: prompt templates per condition.
PROMPTS = {c: None for c in CONDITIONS}


In [ ]:
# TODO: dataset loading. MMLU primary, TriviaQA secondary.
# Hold the item ids fixed across conditions so 03 can split by item.

# Smoke test: set to True to run on 100 items and verify pipeline
SMOKE_TEST = True
N_ITEMS_FULL = 5000
N_ITEMS = 100 if SMOKE_TEST else N_ITEMS_FULL

if SMOKE_TEST:
    print("🔥 SMOKE TEST MODE: Running on 100 items to verify pipeline")
else:
    print("🚀 PRODUCTION MODE: Running on 5000 items")

In [3]:
# Import data utilities
from nandaproj import data
import numpy as np
from tqdm.auto import tqdm

ModuleNotFoundError: No module named 'nandaproj'

## Dataset Loading (no GPU needed)

Load MMLU (primary) and TriviaQA (secondary) from HuggingFace datasets. These run locally, no need to rent the GPU yet.

**Smoke Test:** Set `SMOKE_TEST = True` above to run on 100 items and verify the entire pipeline works before scaling to 5000. Activations cache will be ~18.5 MB instead of 1.85 GB.

In [2]:
# Load MMLU (primary dataset)
print("Loading MMLU validation split...")
mmlu_data = data.load_dataset_mmlu(n_items=N_ITEMS)
print(f"✓ MMLU: {len(mmlu_data['item_ids'])} items from {len(set(mmlu_data['splits']))} subjects")
print(f"  Subjects: {sorted(set(mmlu_data['splits']))[:5]}... ({len(set(mmlu_data['splits']))} total)")
print(f"  Sample: '{mmlu_data['questions'][0]}'")
print(f"  Options: {mmlu_data['options'][0]}")
print(f"  Answer: {mmlu_data['answers'][0]}")

Loading MMLU validation split...


NameError: name 'data' is not defined

In [ ]:
# Load TriviaQA (secondary, for 5.4 transfer check)
print("\nLoading TriviaQA validation split...")
tqa_data = data.load_dataset_triviaqa(n_items=N_ITEMS)
print(f"✓ TriviaQA: {len(tqa_data['item_ids'])} items")
print(f"  Sample: '{tqa_data['questions'][0]}'")
print(f"  Answer: '{tqa_data['answers_text'][0]}'")
print(f"  Aliases: {tqa_data['answer_aliases'][0][:3] if tqa_data['answer_aliases'][0] else 'none'}...")

## Cache Schema & Size Preview

**Decision:** .npz format, one file per (dataset, condition) pair.

- **File:** `results/behavioral_{dataset}_{condition}.npz`
- **Structure:** One row per item, columns for (item_id, answer, correct, confidence, activations_L{layer}, ...)
- **Layers:** Initially layers [12, 14, 16, 18, 20, 22] (last 12 layers, every other; ~1.85 GB total)
- **Full sweep:** All 24 layers (~7.4 GB); can expand after initial results look good

See `DATA_ANALYSIS.md` for full breakdown.

## Smoke Test Size Impact

Per-condition cache size scales with N_ITEMS:

- **100 items (smoke test):** 1.23 MB per condition → **~12.3 MB total** (10 conditions: 5 conditions × 2 datasets)
- **5000 items (production):** 61.5 MB per condition → **~615 MB total**

**Disk I/O time:** ~0.37 sec (100) vs ~18.5 sec (5000) per save

The smoke test is 50× smaller—useful for verifying:
1. Prompt template formatting works
2. Model generation and activation capture works
3. Confidence spread has signal (V2 gate passes)
4. Cache save/load I/O works correctly

Then scale up to 5000 items for the real run.

In [ ]:
# Create empty cache files for each (dataset, condition) pair
print("Creating cache file structure...\n")
if SMOKE_TEST:
    print(f"📋 SMOKE TEST: {N_ITEMS} items per condition\n")
else:
    print(f"📋 PRODUCTION: {N_ITEMS} items per condition\n")

cache_info = {}
for dataset in ["mmlu", "triviaqa"]:
    cache_info[dataset] = {}
    for condition in data.CONDITIONS:
        cache_path = data.create_cache_file(dataset, condition, n_items=N_ITEMS)
        size_mb = cache_path.stat().st_size / 1e6
        cache_info[dataset][condition] = size_mb
        print(f"  {dataset:8s} × {condition:15s}: {size_mb:6.2f} MB")

print("\n" + "="*60)
total_mb = sum(sum(v.values()) for v in cache_info.values())
n_per_condition = len(cache_info) * 2  # 2 datasets
print(f"Total allocated: {total_mb:.1f} MB for {n_per_condition} conditions")
print(f"Per condition:   {total_mb / n_per_condition:.2f} MB")
print(f"Layers per file: {len(data.LAYER_INDICES)} (indices: {data.LAYER_INDICES})")
print(f"Hidden dim:      3072 (gemma-3-4b-it)")
if SMOKE_TEST:
    print(f"Scale-up factor: {N_ITEMS_FULL / N_ITEMS:.0f}× to reach {N_ITEMS_FULL} items")
print("="*60)

In [ ]:
# Preview: what the cache structure looks like (with mock data)
print("\nCache file structure (example with mock data):\n")

# Create a small mock cache to show structure
n_mock = 3
mock_data = {
    "item_ids": np.array([f"mmlu_{i:05d}" for i in range(n_mock)], dtype=object),
    "answers": np.array([0, 2, 1], dtype=np.int32),
    "correct": np.array([True, False, True], dtype=bool),
    "confidence": np.array([0.87, 0.62, 0.91], dtype=np.float32),
}

# Add mock activations for each layer
for layer_idx in data.LAYER_INDICES:
    mock_data[f"activations_L{layer_idx}"] = np.random.randn(n_mock, 3072).astype(np.float32)

print("Keys in .npz file:")
for key, value in mock_data.items():
    if isinstance(value, np.ndarray):
        print(f"  {key:25s} shape={str(value.shape):15s} dtype={str(value.dtype):8s}")

print(f"\nExample row (first item, scaled to {N_ITEMS} items):")
print(f"  item_id:      {mock_data['item_ids'][0]}")
print(f"  answer:       {mock_data['answers'][0]}")
print(f"  correct:      {mock_data['correct'][0]}")
print(f"  confidence:   {mock_data['confidence'][0]:.3f}")
print(f"  activations:  {len(data.LAYER_INDICES)} layers × (3072,) float32 vectors")
print(f"  → {N_ITEMS} rows in the actual cache file")

In [ ]:
# API reference: how to save and load cache during generation
print("Cache I/O API:\n")

print("SAVING during generation loop:")
print("  data.save_cache(")
print("      dataset='mmlu',")
print("      condition='P1_separate',")
print("      item_ids=np.array([...]),        # (N,) str")
print("      answers=np.array([...]),         # (N,) int32")
print("      correct=np.array([...]),         # (N,) bool")
print("      confidence=np.array([...]),      # (N,) float32")
print("      activations={")
print("          12: activations_layer_12,    # (N, 3072) float32")
print("          14: activations_layer_14,    # etc.")
print("          ...",
print("      }")
print("  )")

print("\nLOADING after generation:")
print("  cache = data.load_cache('mmlu', 'P1_separate')")
print("  cache['item_ids']      → (5000,) array")
print("  cache['answers']       → (5000,) array")
print("  cache['correct']       → (5000,) array")
print("  cache['confidence']    → (5000,) array")
print("  cache['activations_L12'] → (5000, 3072) array")
print("  cache['activations_L14'] → (5000, 3072) array")
print("  ... etc for each layer")

In [ ]:
# TODO: generation loop.
#
# This is the long one -- instrument it. Per the global CLAUDE.md, anything over
# five minutes gets a visible progress bar:
#
#     from tqdm.auto import tqdm
#     for item in tqdm(items, desc=f"{cond} ({model_name})"):
#
# Capture residual activations at the answer-adjacent position across the layer
# sweep in the same pass -- re-running generation to collect activations you
# forgot is the single most expensive mistake available here.


In [ ]:
# V2 hard gate. Run this the moment the first condition finishes -- do not wait
# until all five are done to discover the dependent variable is degenerate.

# TODO: replace with real values.
conf = None  # array of verbalized confidences

# spread = conf.std()
# n_unique = len(set(conf.round(2)))
# gate("V2 confidence spread", spread > 0.05 and n_unique >= 5,
#      f"std={spread:.3f}, {n_unique} distinct values")

gate("V2 confidence spread", False, "not implemented yet")


In [ ]:
# TODO: persist to config.RESULTS. Parquet or npz, one file per (dataset,
# condition), activations kept separately from the scalar table.
#
# Save *before* the box dies. `just down` syncs results/ back, but the watchdog
# self-destroys after 45 min idle and does not sync.
